# Method 4, phase 1: the canvas bound on a Colab GPU

Runs one chunk of `train_all` (100 clips) from the `canvas-bound` branch: the Method 3 clip
caches (`--full-scales 1`), then `python -m scripts.method4 bound`, for each model. The bound
file is copied to `MyDrive/avrh/method4/train_all/` every 10 clips and at the end; a runtime
that is cut off can be run again and resumes (bound files already complete on Drive are skipped).

**Needs on Drive, in `MyDrive/avrh/`:** `vps_train_all_chunk{CHUNK}.zip` and
`PIDNet_S_Cityscapes_val.pt` (the same files as the Method 3 notebook).

**To run:** set `CHUNK` in the first cell, then Runtime → Run all. About 2 h 30 for the two
models. Accuracy only, fp32: no latency is measured here.

**Afterwards:** download `bound_train_all_<model>_c{CHUNK}of4.jsonl.gz` (two files, about
150 MB together) into `experiments/method4/train_all/` on the laptop, then `merge` and `score`.

In [ ]:
CHUNK = 2                        # 0..3: vps_train_all_chunk{CHUNK}.zip
MODELS = ["pidnet_s", "bisenetv2"]
BRANCH = "canvas-bound"
DRIVE = "/content/drive/MyDrive/avrh"

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,temperature.gpu --format=csv
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime -> Change runtime type)."
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os, shutil, time, zipfile
from google.colab import drive
drive.mount("/content/drive")
VPS = "/content/vps"
ZIP = f"vps_train_all_chunk{CHUNK}.zip"
marker = f"{VPS}/.{ZIP}_ok"
if not os.path.exists(marker):
    shutil.rmtree(VPS, ignore_errors=True)   # one chunk on disk at a time
    shutil.rmtree("/content/m3cache", ignore_errors=True)
    t = time.time()
    shutil.copy(f"{DRIVE}/{ZIP}", "/content/vps.zip")   # local disk: much faster reads than Drive
    zipfile.ZipFile("/content/vps.zip").extractall(VPS)
    os.remove("/content/vps.zip")
    open(marker, "w").close()
    print(f"{ZIP} ready in {time.time() - t:.0f} s")
shutil.copy(f"{DRIVE}/PIDNet_S_Cityscapes_val.pt", "/content/PIDNet_S_Cityscapes_val.pt")

In [ ]:
!rm -rf /content/avrh && git clone -q --depth 1 -b $BRANCH https://github.com/AntoineBesson/Active-Vision-on-reduced-hardware.git /content/avrh
%cd /content/avrh
!git log --oneline -1
!pip install -q -e ".[reuse]"

In [ ]:
import shlex, subprocess, sys
env = dict(os.environ, CITYSCAPES_VPS_ROOT=VPS, PIDNET_S_WEIGHTS="/content/PIDNet_S_Cityscapes_val.pt")
OUT, ROOT, LOCAL = f"{DRIVE}/method4/train_all", "/content/m3cache", "/content/m4out"
for d in (OUT, ROOT, LOCAL):
    os.makedirs(d, exist_ok=True)
TAG = f"c{CHUNK}of4"   # the name `--chunk {CHUNK}/4` gives on the laptop: the zip holds exactly that block of clips
log = f"/content/method4_{TAG}.log"

def run(module, args, sync=None):
    """Run a CLI stage; ``sync``: a file copied to Drive every 10 clips (bound appends clip by clip)."""
    t, n = time.time(), 0
    with open(log, "a") as f:
        proc = subprocess.Popen([sys.executable, "-u", "-m", module, *shlex.split(args)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in proc.stdout:
            f.write(line)
            print(line, end="", flush=True)
            n += " clip " in line
            if sync and n and n % 10 == 0 and " clip " in line and os.path.exists(sync):
                shutil.copy(sync, OUT)
    if proc.wait() != 0:
        raise RuntimeError(f"{module} {args.split()[0]} failed (exit {proc.returncode}); full log: {log}")
    print(f"== {module} {args.split()[0]} done in {(time.time() - t) / 60:.1f} min ==")

In [ ]:
for MODEL in MODELS:
    name = f"bound_train_all_{MODEL}_{TAG}.jsonl.gz"
    if os.path.exists(f"{OUT}/{name}.done"):
        print("already complete on Drive:", name)
        continue
    if os.path.exists(f"{OUT}/{name}") and not os.path.exists(f"{LOCAL}/{name}"):
        shutil.copy(f"{OUT}/{name}", LOCAL)   # a run that was cut off: bound skips the clips it holds
    run("scripts.method3", f"cache --split train_all --models {MODEL} --root {ROOT} --full-scales 1")
    run("scripts.method4", f"bound --split train_all --models {MODEL} --root {ROOT} --tag {TAG} --out-dir {LOCAL}",
        sync=f"{LOCAL}/{name}")
    shutil.copy(f"{LOCAL}/{name}", OUT)
    open(f"{OUT}/{name}.done", "w").close()
    print("saved to Drive:", f"{OUT}/{name}")
    shutil.rmtree(f"{ROOT}/train_all/{MODEL}", ignore_errors=True)   # 2 GB per model: free the disk
print(sorted(os.listdir(OUT)))